# 🌸 Scrapping Data

## 1. Instalasi Library

Pada tahap awal, dilakukan instalasi library yang digunakan untuk proses web scraping. Library `requests` digunakan untuk mengambil halaman website, sedangkan `beautifulsoup4` digunakan untuk membaca dan mengambil informasi dari struktur HTML halaman website.

In [52]:
%pip install requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


## 2. Import Library

Pada tahap ini, beberapa library di-import untuk membantu proses web scraping:

- `requests` digunakan untuk mengambil halaman website.
- `BeautifulSoup` digunakan untuk membaca dan mengambil data dari struktur HTML.
- `csv` digunakan untuk mengolah atau menyimpan data dalam format CSV.
- `time` digunakan untuk memberikan jeda waktu saat proses scraping.
- `urljoin` digunakan untuk menggabungkan URL agar menjadi link yang lengkap.

In [53]:
import requests
from bs4 import BeautifulSoup
import csv
import time
from urllib.parse import urljoin

## 3. Membuat Header Request

Pada tahap ini, dibuat `HEADERS` yang berisi `User-Agent` untuk memberi informasi bahwa request berasal dari browser seperti Chrome pada Windows. Penggunaan `User-Agent` membantu proses scraping agar request lebih mudah diterima oleh website dan tidak langsung dianggap sebagai request dari bot.

In [ ]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    )
}

## 4. Membuat Session

Pada tahap ini, dibuat `session` menggunakan `requests.Session()` agar koneksi dan pengaturan request dapat digunakan kembali selama proses scraping. Kemudian `HEADERS` yang sudah dibuat sebelumnya diterapkan ke session dengan `session.headers.update(HEADERS)`, sehingga setiap request otomatis menggunakan `User-Agent` tersebut.

In [55]:
session = requests.Session()
session.headers.update(HEADERS)

## 5. Mengambil URL Artikel

Pada tahap ini, dibuat fungsi `ambil_url_artikel()` untuk mengambil URL artikel dari website Detik.com. Fungsi ini mengambil artikel dari halaman utama kategori terlebih dahulu, kemudian dilanjutkan ke halaman indeks jika jumlah artikel yang dibutuhkan belum terpenuhi.

Prosesnya berjalan seperti berikut:

1. Fungsi menerima URL kategori, domain, dan jumlah URL yang ingin diambil.
2. Halaman pertama yang diakses adalah halaman utama kategori. Jika belum mendapatkan jumlah URL yang cukup, fungsi berpindah ke halaman indeks berikutnya.
3. Setiap halaman dibaca menggunakan `BeautifulSoup` untuk mencari semua link (`<a>`) yang tersedia.
4. Link kemudian diubah menjadi URL lengkap menggunakan `urljoin()`.
5. URL disaring berdasarkan domain dan pola `"/d-"` agar yang diambil hanya URL artikel Detik.
6. URL yang sudah pernah ditemukan tidak dimasukkan lagi sehingga tidak terjadi duplikasi.
7. Jika jumlah URL sudah mencapai target, proses pengambilan dihentikan.
8. Setelah selesai membaca satu halaman, program memberikan jeda 1 detik menggunakan `time.sleep(1)` sebelum melanjutkan ke halaman berikutnya.
9. Proses dibatasi maksimal sampai 30 halaman untuk mencegah scraping berjalan terus-menerus jika jumlah artikel yang ditemukan tidak mencukupi.

Hasil akhirnya berupa daftar URL artikel yang jumlahnya disesuaikan dengan nilai `jumlah` yang diberikan.

In [56]:
def ambil_url_artikel(url_kategori, domain, jumlah=100):
    """
    Mengambil URL artikel dari homepage dan
    halaman indeks sampai jumlah URL terpenuhi.
    """

    print(f"\nMengambil artikel dari: {url_kategori}")

    daftar_url = []
    halaman = 1

    while len(daftar_url) < jumlah:

        if halaman == 1:
            url = url_kategori
        else:
            url = f"{url_kategori.rstrip('/')}/indeks?page={halaman}"

        print(f"\nMengambil halaman {halaman}:")
        print(url)

        try:
            response = session.get(
                url,
                timeout=15
            )

            print("Status:", response.status_code)

        except requests.RequestException as error:
            print("Error:", error)
            halaman += 1
            continue

        if response.status_code != 200:
            print("Halaman gagal diakses.")
            halaman += 1
            continue

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        jumlah_sebelum = len(daftar_url)

        for link in soup.find_all("a", href=True):

            href = link["href"].strip()

            href = urljoin(url, href)

            if (
                domain in href
                and "/d-" in href
            ):

                href = href.split("#")[0]

                if href not in daftar_url:

                    daftar_url.append(href)

                    print(
                        f"  [{len(daftar_url)}/{jumlah}] "
                        f"{href}"
                    )

                if len(daftar_url) >= jumlah:
                    break

        jumlah_baru = len(daftar_url) - jumlah_sebelum

        print(
            f"Artikel baru dari halaman ini: "
            f"{jumlah_baru}"
        )

        if jumlah_baru == 0:
            print(
                "Tidak ada artikel baru di halaman ini."
            )

        halaman += 1

        time.sleep(1)

        if halaman > 30:
            print("Batas halaman tercapai.")
            break

    print("\n======================================")
    print(
        f"Total URL {domain}: "
        f"{len(daftar_url)}"
    )
    print("======================================")

    return daftar_url[:jumlah]

## 6. Menguji Pengambilan URL Artikel Sport

Pada tahap ini, URL halaman utama Sport Detik disimpan ke dalam variabel `sport_url`. Selanjutnya, fungsi `ambil_url_artikel()` digunakan untuk mengambil 5 URL artikel dari kategori Sport dengan domain `sport.detik.com`. Data URL tersebut disimpan di dalam variabel `url_test` untuk mengecek apakah proses pengambilan artikel sudah berjalan sesuai yang diharapkan.

In [57]:
sport_url = "https://sport.detik.com/"

In [58]:
url_test = ambil_url_artikel(
    sport_url,
    "sport.detik.com",
    5
)


Mengambil artikel dari: https://sport.detik.com/

Mengambil halaman 1:
https://sport.detik.com/
Status: 200
  [1/5] https://sport.detik.com/raket/d-8653585/asian-games-2026-putri-kw-pede-dengan-komposisi-beregu-putri
  [2/5] https://sport.detik.com/basket/d-8649812/derrick-michael-wujudkan-mimpi-masa-kecil-tampil-di-asian-games-2026
  [3/5] https://sport.detik.com/moto-gp/d-8652425/dokter-perkirakan-kondisi-lengan-marc-marquez-sekitar-50-persen
  [4/5] https://sport.detik.com/raket/d-8653493/tekad-alwi-farhan-lampaui-batas-di-asian-games-2026
  [5/5] https://sport.detik.com/moto-gp/d-8650852/jadwal-motogp-san-marino-2026-misi-marc-marquez-kejar-jorge-martin
Artikel baru dari halaman ini: 5

Total URL sport.detik.com: 5


## 7. Mengambil Isi Berita

Pada tahap ini, dibuat fungsi `ambil_isi_berita()` yang digunakan untuk mengambil isi teks dari halaman artikel Detik berdasarkan URL yang sudah didapatkan sebelumnya. Fungsi ini juga membersihkan bagian-bagian halaman yang tidak diperlukan supaya data yang diperoleh hanya berupa isi berita.

Proses yang dilakukan dalam fungsi ini yaitu:

1. **Membuka halaman artikel**  
   URL artikel diakses menggunakan `session.get()` dengan batas waktu 15 detik. Jika halaman tidak bisa dibuka atau statusnya bukan `200`, fungsi akan mengembalikan data kosong.

2. **Mencari bagian isi berita**  
   Setelah halaman berhasil dibuka, `BeautifulSoup` digunakan untuk membaca struktur HTML. Program kemudian mencoba beberapa selector seperti `.detail__body-text` dan `.detail__body` untuk menemukan bagian yang berisi artikel.

3. **Membersihkan isi artikel**  
   Elemen yang tidak diperlukan seperti `script`, `style`, `iframe`, gambar, video, dan tombol dihapus menggunakan `decompose()`. Hal ini dilakukan agar elemen-elemen tersebut tidak ikut masuk ke dalam data berita.

4. **Mengambil dan merapikan teks**  
   Teks dari bagian artikel diambil menggunakan `get_text()`. Setelah itu, spasi yang berlebihan dirapikan sehingga isi berita menjadi lebih bersih dan mudah digunakan sebagai dataset.

Terakhir, fungsi mengembalikan isi berita yang sudah dibersihkan. Jika terjadi error saat mengambil halaman, fungsi akan mengembalikan data kosong dan proses scraping dapat dilanjutkan ke artikel berikutnya.

In [59]:
def ambil_isi_berita(url):
    """
    Membuka halaman artikel dan mengambil isi berita.
    """

    try:

        response = session.get(
            url,
            timeout=15
        )

        if response.status_code != 200:

            print(
                "Gagal membuka:",
                response.status_code
            )

            return ""

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        selector_list = [
            ".detail__body-text",
            ".detail__body",
            "div.detail__body-text",
            "article"
        ]

        artikel = None

        for selector in selector_list:

            artikel = soup.select_one(selector)

            if artikel is not None:
                break

        if artikel is None:
            return ""

        for tag in artikel.find_all(
            [
                "script",
                "style",
                "iframe",
                "img",
                "video",
                "figure",
                "button"
            ]
        ):

            tag.decompose()

        isi = artikel.get_text(
            " ",
            strip=True
        )

        isi = " ".join(
            isi.split()
        )

        return isi

    except requests.RequestException as error:

        print(
            "Error:",
            error
        )

        return ""

## 8. Menyiapkan Dataset

Pada tahap ini, disiapkan tempat untuk menyimpan seluruh data berita yang nantinya akan dikumpulkan. Variabel `dataset` berupa list kosong yang akan digunakan untuk menampung data setiap artikel, sedangkan `id_data` digunakan sebagai nomor ID untuk setiap data dan dimulai dari angka 1.

In [60]:
# Menyiapkan list untuk dataset akhir
dataset = []

# ID data dimulai dari 1
id_data = 1

## 9. Mengambil 100 Berita Sport

Pada tahap ini, URL berita Sport yang sudah dikumpulkan sebelumnya diproses satu per satu untuk mengambil isi beritanya. Program akan terus mengambil berita sampai mendapatkan 100 artikel yang berhasil diproses.

Alurnya yaitu:

1. `jumlah_sport` digunakan untuk menghitung berapa berita Sport yang sudah berhasil diambil.
2. Program membaca URL berita dari `data_sport` satu per satu.
3. Isi berita diambil menggunakan fungsi `ambil_isi_berita()`.
4. Jika isi berita berhasil diambil, data disimpan ke dalam `dataset` dengan tiga informasi:
   - `id` sebagai nomor data.
   - `isi_berita` sebagai teks berita.
   - `label` dengan nilai `"sport"`.
5. Setiap berita yang berhasil akan menambah `jumlah_sport` dan `id_data`.
6. Progress ditampilkan setiap 25 berita agar bisa melihat perkembangan proses scraping.
7. Jika isi berita gagal diambil, URL tersebut dilewati dan program lanjut ke berita berikutnya.
8. Setelah jumlah berita berhasil mencapai 100, proses dihentikan.

Dengan proses ini, data Sport yang masuk ke dataset hanya berasal dari artikel yang isi beritanya berhasil diambil.

In [61]:
# Menghitung jumlah berita Sport yang berhasil
jumlah_sport = 0

# Memproses URL berita Sport
for berita in data_sport:

    # Berhenti jika sudah mendapatkan 100 berita
    if jumlah_sport >= 100:
        break

    # Mengambil isi berita
    isi = ambil_isi_berita(
        berita["link"]
    )

    # Jika isi berita berhasil diambil
    if isi:

        data = {
            "id": id_data,
            "isi_berita": isi,
            "label": "sport"
        }

        # Memasukkan data ke dataset
        dataset.append(data)

        # Menambah jumlah berita berhasil
        jumlah_sport = jumlah_sport + 1

        # Menambah ID
        id_data = id_data + 1

        # Menampilkan progres setiap 25 data
        if jumlah_sport % 25 == 0:
            print(
                "Sport berhasil:",
                jumlah_sport,
                "/ 100"
            )

    else:

        # Jika gagal, lanjut ke URL berikutnya
        print("Sport gagal, lanjut ke berita berikutnya...")


# Menampilkan hasil Sport
print("Jumlah berita Sport berhasil:", jumlah_sport)

Sport gagal, lanjut ke berita berikutnya...
Sport gagal, lanjut ke berita berikutnya...
Sport gagal, lanjut ke berita berikutnya...
Sport gagal, lanjut ke berita berikutnya...
Sport gagal, lanjut ke berita berikutnya...
Sport berhasil: 25 / 100
Sport gagal, lanjut ke berita berikutnya...
Sport gagal, lanjut ke berita berikutnya...
Sport gagal, lanjut ke berita berikutnya...
Sport gagal, lanjut ke berita berikutnya...
Sport gagal, lanjut ke berita berikutnya...
Sport gagal, lanjut ke berita berikutnya...
Sport gagal, lanjut ke berita berikutnya...
Sport gagal, lanjut ke berita berikutnya...
Sport gagal, lanjut ke berita berikutnya...
Sport gagal, lanjut ke berita berikutnya...
Sport gagal, lanjut ke berita berikutnya...
Sport gagal, lanjut ke berita berikutnya...
Sport gagal, lanjut ke berita berikutnya...
Sport gagal, lanjut ke berita berikutnya...
Sport gagal, lanjut ke berita berikutnya...
Sport gagal, lanjut ke berita berikutnya...
Sport berhasil: 50 / 100
Sport berhasil: 75 / 100
S

In [62]:
url_contoh = url_test[0]

isi_contoh = ambil_isi_berita(url_contoh)

print(isi_contoh)

Jakarta - Putri Kusuma Wardani percaya diri dengan komposisi beregu putri yang dipilih PBSI untuk Asian Games 2026. Dia juga menyebut chemistry satu sama lain sudah klop. Bulutangkis Indonesia mengirimkan 20 wakil untuk tampil di Aichi-Nagoya, 19 September hingga 4 Oktober mendatang. 10 Atlet di antaranya merupakan sektor beregu putri. Yaitu Putri KW, Thalita R. Wiryawan, Ni Kadek Dhinda, Mutiara Ayu Puspitasari, Rachel Allessya Rose, Febi Setianingrum, Febriana Dwipuji Kusuma, Meilysa Trias Puspitasari, Siti Fadia Silva Ramadhanti, dan Nita Violina Marwah. SCROLL TO CONTINUE WITH CONTENT Baca juga: Tekad Alwi Farhan Lampaui Batas di Asian Games 2026 Putri menilai komposisi skuad bulutangkis Indonesia yang mengombinasikan pemain senior dan junior, tidak menemukan kendala dalam membangun koneksi di dalam tim. ADVERTISEMENT Pengalaman berlaga bersama dalam beberapa kejuaraan beregu sebelumnya dinilai telah membentuk kekompakan antar-atlet. "Dengan tim komposisi yang ada cukup kuat sih. K

## 10. Mengecek Hasil Dataset

Pada tahap ini, data yang sudah dikumpulkan diubah menjadi `DataFrame` menggunakan `pandas` agar lebih mudah dilihat dan diolah.

Setelah itu, program menampilkan jumlah seluruh data menggunakan `len(df)` dan menghitung jumlah data berdasarkan label menggunakan `value_counts()`. Dari hasil ini, dapat diketahui apakah jumlah berita Sport dan Finance sudah sesuai dengan target yang ditentukan.

In [63]:
# Mengubah dataset menjadi DataFrame
df = pd.DataFrame(dataset)

# Menampilkan jumlah total data
print("Jumlah total data:", len(df))

# Menampilkan jumlah masing-masing label
print(df["label"].value_counts())

Jumlah total data: 100
label
sport    100
Name: count, dtype: int64


Pada tahap ini, dataset diubah menjadi `DataFrame` menggunakan `pd.DataFrame(dataset)`. Setelah itu, `len(df)` digunakan untuk menghitung jumlah seluruh data yang sudah terkumpul. Hasilnya ditampilkan untuk memastikan jumlah data yang didapat sudah sesuai dengan target.

In [64]:
# Mengubah dataset menjadi DataFrame
df = pd.DataFrame(dataset)

# Menampilkan jumlah data
print("Jumlah total data:", len(df))


Jumlah total data: 100


## 11. Membuat Fungsi Crawling

Pada tahap ini, dibuat fungsi `crawling_kategori()` untuk menggabungkan proses pengambilan URL dan isi berita menjadi satu alur. Fungsi ini bisa digunakan untuk beberapa kategori berita, seperti Sport dan Finance, sehingga tidak perlu membuat kode yang sama berulang kali.

Alurnya yaitu:

1. Fungsi menerima URL kategori, domain, label berita, dan jumlah berita yang ingin diambil.
2. Fungsi `ambil_url_artikel()` dipanggil untuk mendapatkan daftar URL sesuai jumlah yang ditentukan.
3. Setiap URL diproses satu per satu menggunakan fungsi `ambil_isi_berita()`.
4. Jika isi berita berhasil diambil, teks berita dan label dimasukkan ke dalam list `data`.
5. Program menampilkan progres setiap artikel yang sedang diproses, termasuk jumlah karakter isi berita.
6. Jika isi berita tidak ditemukan, data tersebut dilewati dan proses dilanjutkan ke URL berikutnya.
7. `time.sleep(1)` digunakan untuk memberi jeda 1 detik setelah memproses setiap artikel.
8. Proses berhenti jika jumlah berita yang berhasil sudah mencapai target.

Setelah selesai, fungsi mengembalikan `data` yang berisi isi berita beserta label kategorinya. Dengan fungsi ini, proses crawling Sport dan Finance nantinya bisa dilakukan dengan cara yang lebih praktis.

In [ ]:

# Code Cell — Fungsi Crawling

def crawling_kategori(
    url_kategori,
    domain,
    label,
    jumlah=100
):

    print("\n======================================")
    print(f"CRAWLING {label.upper()}")
    print("Target:", jumlah, "berita")
    print("======================================")

    urls = ambil_url_artikel(
        url_kategori,
        domain,
        jumlah
    )

    print(
        f"\nURL yang akan diproses: "
        f"{len(urls)}"
    )

    data = []

    for i, url in enumerate(
        urls,
        start=1
    ):

        print(
            f"\n[{i}/{len(urls)}] "
            f"Mengambil berita {label}..."
        )

        isi = ambil_isi_berita(url)

        if isi:

            data.append({
                "isi_berita": isi,
                "label": label
            })

            print(
                "✓ Berhasil | "
                f"{len(isi)} karakter"
            )

        else:

            print(
                "✗ Isi berita tidak ditemukan"
            )

        time.sleep(1)

        if len(data) >= jumlah:
            break

    print("\n======================================")
    print(
        f"Berita {label} berhasil: "
        f"{len(data)}"
    )
    print("======================================")

    return data

## 12. Menjalankan Proses Crawling

### 12.1 Menentukan URL Kategori

Pada tahap ini, ditentukan URL halaman utama untuk kategori Sport dan Finance pada Detik.com. URL tersebut disimpan ke dalam variabel `sport_url` dan `finance_url` yang nantinya digunakan sebagai alamat awal dalam proses crawling berita.


In [66]:
sport_url = "https://sport.detik.com/"
finance_url = "https://finance.detik.com/"

### 12.2 Crawling Berita Sport

Pada tahap ini, dilakukan pengujian proses crawling untuk kategori Sport dengan target sebanyak 5 berita. Fungsi `crawling_kategori()` digunakan dengan memasukkan URL Sport, domain `sport.detik.com`, label `sport`, dan jumlah berita yang ingin diambil sebanyak 5. Hasil crawling disimpan ke dalam variabel `data_sport` untuk digunakan pada tahap berikutnya.

In [67]:
data_sport = crawling_kategori(
    sport_url,
    "sport.detik.com",
    "sport",
    5
)


CRAWLING SPORT
Target: 5 berita

Mengambil artikel dari: https://sport.detik.com/

Mengambil halaman 1:
https://sport.detik.com/
Status: 200
  [1/5] https://sport.detik.com/raket/d-8653585/asian-games-2026-putri-kw-pede-dengan-komposisi-beregu-putri
  [2/5] https://sport.detik.com/basket/d-8649812/derrick-michael-wujudkan-mimpi-masa-kecil-tampil-di-asian-games-2026
  [3/5] https://sport.detik.com/moto-gp/d-8652425/dokter-perkirakan-kondisi-lengan-marc-marquez-sekitar-50-persen
  [4/5] https://sport.detik.com/raket/d-8653493/tekad-alwi-farhan-lampaui-batas-di-asian-games-2026
  [5/5] https://sport.detik.com/moto-gp/d-8650852/jadwal-motogp-san-marino-2026-misi-marc-marquez-kejar-jorge-martin
Artikel baru dari halaman ini: 5

Total URL sport.detik.com: 5

URL yang akan diproses: 5

[1/5] Mengambil berita sport...
✓ Berhasil | 2720 karakter

[2/5] Mengambil berita sport...
✓ Berhasil | 2749 karakter

[3/5] Mengambil berita sport...
✓ Berhasil | 2191 karakter

[4/5] Mengambil berita sport.

### 12.3 Crawling Berita Finance

Pada tahap ini, dilakukan pengujian proses crawling untuk kategori Finance dengan target sebanyak 5 berita. Fungsi `crawling_kategori()` digunakan dengan URL Finance, domain `finance.detik.com`, label `finance`, dan jumlah berita sebanyak 5. Hasil crawling disimpan ke dalam variabel `data_finance` untuk digunakan pada tahap berikutnya.

In [68]:
data_finance = crawling_kategori(
    finance_url,
    "finance.detik.com",
    "finance",
    5
)


CRAWLING FINANCE
Target: 5 berita

Mengambil artikel dari: https://finance.detik.com/

Mengambil halaman 1:
https://finance.detik.com/
Status: 200
  [1/5] https://finance.detik.com/moneter/d-8655958/duit-apbn-dipakai-biayai-buka-rekening-bank-purbaya-buka-suara
  [2/5] https://finance.detik.com/moneter/d-8655766/masyarakat-dibuatkan-rekening-saldo-awal-rp-50-ribu-sumber-apbn-rp-11-t
  [3/5] https://finance.detik.com/berita-ekonomi-bisnis/d-8655007/pm-singapura-sumbangkan-kenaikan-gaji-rp-24-m-selama-5-tahun
  [4/5] https://finance.detik.com/berita-ekonomi-bisnis/d-8654941/harga-emas-antam-makin-jatuh
  [5/5] https://finance.detik.com/berita-ekonomi-bisnis/d-8654769/ratusan-penerbangan-di-inggris-delay-batal-ada-apa
Artikel baru dari halaman ini: 5

Total URL finance.detik.com: 5

URL yang akan diproses: 5

[1/5] Mengambil berita finance...
✓ Berhasil | 1707 karakter

[2/5] Mengambil berita finance...
✓ Berhasil | 1468 karakter

[3/5] Mengambil berita finance...
✓ Berhasil | 2431 karak

### 12.4 Crawling 100 Berita Sport

Setelah proses pengujian berhasil dilakukan, tahap berikutnya adalah mengambil data utama untuk kategori Sport sebanyak 100 berita. Fungsi `crawling_kategori()` digunakan kembali dengan target 100 berita, kemudian hasil crawling disimpan ke dalam variabel `data_sport`.

In [ ]:
data_sport = crawling_kategori(
    sport_url,
    "sport.detik.com",
    "sport",
    100
)


CRAWLING SPORT
Target: 100 berita

Mengambil artikel dari: https://sport.detik.com/

Mengambil halaman 1:
https://sport.detik.com/
Status: 200
  [1/100] https://sport.detik.com/raket/d-8653585/asian-games-2026-putri-kw-pede-dengan-komposisi-beregu-putri
  [2/100] https://sport.detik.com/basket/d-8649812/derrick-michael-wujudkan-mimpi-masa-kecil-tampil-di-asian-games-2026
  [3/100] https://sport.detik.com/moto-gp/d-8652425/dokter-perkirakan-kondisi-lengan-marc-marquez-sekitar-50-persen
  [4/100] https://sport.detik.com/raket/d-8653493/tekad-alwi-farhan-lampaui-batas-di-asian-games-2026
  [5/100] https://sport.detik.com/moto-gp/d-8650852/jadwal-motogp-san-marino-2026-misi-marc-marquez-kejar-jorge-martin
  [6/100] https://sport.detik.com/sport-lain/d-8655872/pelatih-timnas-voli-putra-ri-targetkan-medali-di-asian-games-2026
  [7/100] https://sport.detik.com/sport-lain/d-8656039/ketum-koi-memastikan-atlet-terlayani-dengan-baik-di-nagoya
  [8/100] https://sport.detik.com/sport-lain/d-865577

### 12.5 Crawling 100 Berita Finance

Setelah proses pengujian dilakukan, tahap berikutnya adalah mengambil data utama untuk kategori Finance sebanyak 100 berita. Fungsi `crawling_kategori()` digunakan kembali dengan target 100 berita, kemudian hasil crawling disimpan ke dalam variabel `data_finance`.

In [70]:
data_finance = crawling_kategori(
    finance_url,
    "finance.detik.com",
    "finance",
    100
)


CRAWLING FINANCE
Target: 100 berita

Mengambil artikel dari: https://finance.detik.com/

Mengambil halaman 1:
https://finance.detik.com/
Status: 200
  [1/100] https://finance.detik.com/moneter/d-8655958/duit-apbn-dipakai-biayai-buka-rekening-bank-purbaya-buka-suara
  [2/100] https://finance.detik.com/moneter/d-8655766/masyarakat-dibuatkan-rekening-saldo-awal-rp-50-ribu-sumber-apbn-rp-11-t
  [3/100] https://finance.detik.com/berita-ekonomi-bisnis/d-8655007/pm-singapura-sumbangkan-kenaikan-gaji-rp-24-m-selama-5-tahun
  [4/100] https://finance.detik.com/berita-ekonomi-bisnis/d-8654941/harga-emas-antam-makin-jatuh
  [5/100] https://finance.detik.com/berita-ekonomi-bisnis/d-8654769/ratusan-penerbangan-di-inggris-delay-batal-ada-apa
  [6/100] https://finance.detik.com/moneter/d-8656071/kapan-pembukaan-rekening-buat-masyarakat-dimulai-ini-jawabannya
  [7/100] https://finance.detik.com/moneter/d-8655862/bri-ditugaskan-buka-rekening-warga-ri-khusus-aceh-lewat-bsi
  [8/100] https://finance.deti

In [71]:
semua_data = (
    data_sport +
    data_finance
)

print("Total data:", len(semua_data))

Total data: 200


### 12.6 Menyimpan Dataset ke File CSV

Pada tahap ini, seluruh data berita yang sudah dikumpulkan disimpan ke dalam file CSV agar dapat digunakan pada tahap pengolahan data berikutnya. File dibuat dengan nama `dataset_berita.csv` dan memiliki tiga kolom, yaitu `id`, `isi_berita`, dan `label`.

Data pada `semua_data` kemudian ditulis satu per satu ke dalam file CSV. Nomor `id` dibuat secara berurutan mulai dari 1, sedangkan isi berita dan label diambil dari masing-masing data. Setelah proses selesai, program menampilkan pesan bahwa dataset berhasil disimpan.

In [72]:
with open(
    "dataset_berita.csv",
    "w",
    newline="",
    encoding="utf-8-sig"
) as file:

    writer = csv.writer(file)

    writer.writerow([
        "id",
        "isi_berita",
        "label"
    ])

    for i, berita in enumerate(
        semua_data,
        start=1
    ):

        writer.writerow([
            i,
            berita["isi_berita"],
            berita["label"]
        ])

print("Dataset berhasil disimpan sebagai dataset_berita.csv")

Dataset berhasil disimpan sebagai dataset_berita.csv


## 13. Membaca dan Mengecek Dataset

### 13.1 Instalasi Library Pandas

Pada tahap ini, library `pandas` diinstal untuk membantu membaca, mengolah, dan mengecek dataset yang sudah disimpan dalam format CSV.

In [ ]:
%pip install pandas

### 13.2 Membaca Dataset

Pada tahap ini, dataset hasil crawling yang telah disimpan dalam file `dataset_berita.csv` dibaca menggunakan library `pandas`. Data tersebut disimpan ke dalam variabel `df` dalam bentuk DataFrame. Setelah itu, `len(df)` digunakan untuk mengetahui jumlah seluruh data berita yang terdapat dalam dataset.

In [74]:
import pandas as pd

# Membaca dataset hasil crawling
df = pd.read_csv("dataset_berita.csv")

print("Jumlah seluruh data:", len(df))

Jumlah seluruh data: 200


### 13.3 Menampilkan Data Sport dan Finance

Pada tahap ini, ditampilkan masing-masing 5 data dari kategori Sport dan Finance untuk mengecek isi dataset yang sudah diperoleh. Data difilter berdasarkan nilai pada kolom `label`, kemudian hanya kolom `id`, `isi_berita`, dan `label` yang ditampilkan.

In [75]:
print("===== 5 DATA SPORT =====")

display(
    df[df["label"] == "sport"][
        ["id", "isi_berita", "label"]
    ].head(5)
)

print("\n===== 5 DATA FINANCE =====")

display(
    df[df["label"] == "finance"][
        ["id", "isi_berita", "label"]
    ].head(5)
)

===== 5 DATA SPORT =====


,id,isi_berita,label
0,1,Jakarta - Putri Kusuma Wardani percaya diri de...,sport
1,2,Jakarta - Mata Derrick Michael Xzavierro berbi...,sport
2,3,Jakarta - Marc Marquez masih harus beradaptasi...,sport
3,4,Jakarta - Momen debut di Asian Games 2026 tak ...,sport
4,5,Jakarta - MotoGP 2026 akan berlanjut ke San Ma...,sport



===== 5 DATA FINANCE =====


,id,isi_berita,label
100,101,Jakarta - Menteri Keuangan (Menkeu) Purbaya Yu...,finance
101,102,Jakarta - Semua masyarakat akan memiliki reken...,finance
102,103,"Jakarta - Perdana Menteri (PM) Singapura, Lawr...",finance
103,104,Jakarta - Harga emas Antam terus mengalami pen...,finance
104,105,"Jakarta - Penyedia kontrol lalu lintas udara, ...",finance
